# P12 — Čujni aliasing: ton i klizni ton uzorkovani preniskom frekvencijom

Bilježnica uz predavanje P12 i **poglavlje 12 udžbenika *Signali i sustavi*** (Uzorkovanje i
diskretni signali). Ton zadane frekvencije $f$ uzorkujemo frekvencijom $f_s$ i uzorke pustimo
kao zvuk. Dok je $f$ ispod Nyquistove frekvencije $f_s/2$, čuje se isti ton. Iznad nje uzorci
nose nižu, *prividnu* frekvenciju $f_a$, pa se čuje drugi, dublji ton: aliasing (preklapanje
spektra) se čuje. Klizačima se biraju $f$ i $f_s$; graf savijanja pokazuje kamo se $f$
preslika. Klizni ton (engl. *chirp*) jednoliko raste, a nakon uzorkovanja iznad $f_s/2$
„pada natrag”. Za usporedbu
se isti zvuk prije uzorkovanja propušta kroz anti-aliasing filtar.

**Kako pokrenuti.** Ćelije pokrenite redom (Shift + Enter) ili izbornikom *Run → Run All
Cells*. Instalaciju Pythona i JupyterLaba opisuje `README.md` repozitorija. Bilježnica radi bez
interneta, a sve zvukove sama sintetizira. Za slušanje trebaju zvučnici ili slušalice; bez njih
rade grafovi i ispis. Glasnoću na računalu najprije smanjite: visoki tonovi su neugodni. Ako se
klizač ne prikaže, funkciju pokusa pozovite izravno (odjeljak 3).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy import signal
from IPython.display import Audio, display
import ipywidgets as widgets

BOJA = "#840B55"                       # bordo: prividna frekvencija i alias
SIVA, TAMNA = "#5B6770", "#222222"
# Krivulje se razlikuju i vrstom crte, pa su čitljive i crno-bijelo.

# decimalni zarez i pravi minus na osima
zarez = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ",").replace("-", "−"))
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3})

## Postavke

„Analogni” zvuk ovdje je zadan uzorcima uz $F_S = 48$ kHz. Njegova Nyquistova frekvencija,
$24$ kHz, iznad je svega što čujemo, pa ga za uho možemo smatrati kontinuiranim. Uzorkovanje
frekvencijom $f_s$ tada je uzimanje svakog $M$-tog uzorka, $M = F_S/f_s$ (zato $f_s$ na klizaču
dijeli $F_S$). Uzorci se za slušanje vraćaju na $F_S$ interpolacijom niskopropusnim filtrom do
$f_s/2$, kao u D/A pretvorniku s izlaznim filtrom (poglavlje 12), pa se čuje upravo ono što
uzorci nose. Druge vrijednosti zadajte u ćeliji ispod, pa pokrenite sve ćelije.

In [ ]:
# ===== POSTAVKE (mijenjajte po želji, pa Run → Run All Cells) =====
FS = 48000               # Hz: „analogni” zvuk i reprodukcija (F_S)
TRAJANJE = 2.0           # s: trajanje tona
GLASNOCA = 0.3           # amplituda tona, isti faktor za sve zvukove
F_MIN = 100              # Hz: najniža frekvencija tona na klizaču
F_MAX = 16000            # Hz: najviša frekvencija tona na klizaču
F_KORAK = 50             # Hz: korak klizača f
F_POCETNA = 4400         # Hz: frekvencija tona nakon pokretanja
FS_IZBOR = (2400, 3000, 4800, 6000, 9600, 12000, 16000, 24000)   # Hz: f_s na klizaču
FS_POCETNA = 4800        # Hz: f_s nakon pokretanja
FILTAR_RED = 8           # red anti-aliasing filtra (Butterworthov niskopropusni)
FILTAR_UDIO = 0.9        # lom filtra je FILTAR_UDIO · f_s/2
N_PRIKAZ = 24            # broj uzoraka na grafu u vremenu
CHIRP_OD = 100           # Hz: početna frekvencija kliznog tona
CHIRP_DO = 12000         # Hz: završna frekvencija kliznog tona
CHIRP_TRAJANJE = 6.0     # s: trajanje kliznog tona
FS_CHIRP = 6000          # Hz: f_s za klizni ton (mora dijeliti FS)

# Iz knjige (poglavlje 12). Primjer savijanja frekvencije: f_s i prividne frekvencije za
# frekvencije signala iz primjera (u knjizi u Hz, ovdje u kHz; odjeljak 2). Primjer izbora f_s
# i anti-aliasing filtra: f_s, smetnja i lom kaskade dvaju RC krugova [Hz], prividna frekvencija
# smetnje [Hz] i pojačanje kaskade na smetnji [dB] (odjeljak 5).
SAVIJANJE_FS_KHZ = 10
SAVIJANJE_FA_KNJIGA = {2: 2, 6: 4, 8: 2, 12: 2, 15: 5, 18: 2, 23: 3}
IZBOR_FS = 3000
IZBOR_FD = 2950
IZBOR_LOM = 250
IZBOR_FA_KNJIGA = 50
IZBOR_PRIGUSENJE_KNJIGA = -42.9

## 1. Prividna frekvencija

Uzorkujemo li ton $\cos 2\pi f t$ frekvencijom $f_s$, dobivamo niz
$x[n] = \cos \varOmega n$ s frekvencijom $\varOmega = 2\pi f / f_s$ rad/uzorak. Prema teoremu o
frekvencijama koje se razlikuju za $2\pi$ (poglavlje 12) iste uzorke daju frekvencije
$f + k f_s$ za svaki cijeli $k$, a za kosinus i $f_s - f$. Zato su uzorci tona frekvencije $f$ jednaki
uzorcima tona *prividne frekvencije*

$$f_a = |f - k f_s| ,\qquad k\ \text{cijeli broj najbliži}\ f/f_s ,\qquad
0 \le f_a \le \frac{f_s}{2}$$

(teorem o prividnoj frekvenciji). Tonove do
Nyquistove frekvencije $f_s/2$ uzorkovanje ne mijenja, a sve iznad nje *savija* u interval
$[0, f_s/2]$. Iz uzoraka se više ne može saznati od koje je frekvencije niz nastao, pa se
aliasing nakon uzorkovanja ne može ispraviti. Komponentu iznad $f_s/2$ treba ukloniti *prije*
uzorkovanja, analognim niskopropusnim **anti-aliasing filtrom**. Ovdje ga oponaša
Butterworthov filtar reda `FILTAR_RED` s lomom $0{,}9 \cdot f_s/2$, koji djeluje na „analogni”
zvuk.

Funkcije u ćeliji ispod računaju prividnu frekvenciju, ton i klizni ton, uzorkovanje s filtrom i
bez njega, zvuk iz uzoraka i ispis. Ispis slijedi postupak za prividnu frekvenciju iz
poglavlja 12.

In [ ]:
RUB_ZVUKA = 0.02         # s: meki početak i kraj zvuka (bez klika)


def broj(v, decimala=0):
    # broj s decimalnim zarezom i pravim minusom (−42,9); −0 piše se 0
    tekst = f"{v:.{decimala}f}"
    if float(tekst) == 0:
        tekst = tekst.lstrip("-")
    return tekst.replace(".", ",").replace("-", "−")


def prividna(f, fs):
    # (k, r, f_a) kao u poglavlju 12: k je cijeli broj najbliži f/f_s (pola prema gore),
    # r = f − k·f_s, a prividna frekvencija f_a = |r| leži u [0, f_s/2]
    k = int(np.floor(f / fs + 0.5))
    r = f - k * fs
    return k, r, abs(r)


def prividna_niz(f, fs):
    # f_a za niz frekvencija f (za grafove), isti račun kao prividna
    f = np.asarray(f, dtype=float)
    return np.abs(f - np.floor(f / fs + 0.5) * fs)


def korak(fs):
    # M = F_S/f_s: uzorkovanje s f_s uzima svaki M-ti uzorak „analognog” zvuka
    if FS % fs:
        raise ValueError(f"f_s = {fs} Hz ne dijeli FS = {FS} Hz; izaberite f_s iz FS_IZBOR")
    return int(FS // fs)


def ton(f, trajanje=TRAJANJE):
    # „analogni” ton GLASNOCA·cos(2πft), zadan uzorcima uz F_S
    t = np.arange(int(trajanje * FS)) / FS
    return GLASNOCA * np.cos(2 * np.pi * f * t)


def frekvencija_kliznog(t):
    # trenutačna frekvencija kliznog tona: raste jednoliko od CHIRP_OD do CHIRP_DO
    return CHIRP_OD + (CHIRP_DO - CHIRP_OD) * np.asarray(t, dtype=float) / CHIRP_TRAJANJE


def klizni_ton():
    # klizni ton (chirp) amplitude GLASNOCA, zadan uzorcima uz F_S
    t = np.arange(int(CHIRP_TRAJANJE * FS)) / FS
    return GLASNOCA * signal.chirp(t, CHIRP_OD, CHIRP_TRAJANJE, CHIRP_DO, method="linear")


def lom_filtra(fs):
    return FILTAR_UDIO * fs / 2


def antialias(fs):
    # anti-aliasing filtar za f_s: Butterworthov niskopropusni filtar reda FILTAR_RED s lomom
    # FILTAR_UDIO·f_s/2; djeluje na „analogni” zvuk, dakle prije uzorkovanja
    return signal.butter(FILTAR_RED, lom_filtra(fs), fs=FS, output="sos")


def pojacanje_db(fs, f):
    # pojačanje anti-aliasing filtra za f_s na frekvenciji f (ili nizu frekvencija), u dB
    _, h = signal.sosfreqz(antialias(fs), worN=np.atleast_1d(f), fs=FS)
    db = 20 * np.log10(np.maximum(np.abs(h), 1e-12))
    return float(db[0]) if np.ndim(f) == 0 else db


def rc_kaskada_db(f, lom):
    # pojačanje kaskade dvaju RC krugova s lomom lom, 1/(1 + jf/lom)², u dB (poglavlje 12)
    return -20 * np.log10(1 + (np.asarray(f, dtype=float) / lom) ** 2)


def uzorkuj(x, fs, filtar=False):
    # uzorci x[n] uz f_s: svaki M-ti uzorak „analognog” zvuka, bez filtra ili nakon
    # anti-aliasing filtra
    if filtar:
        x = signal.sosfilt(antialias(fs), x)
    return x[::korak(fs)]


def rekonstruiraj(uzorci, fs, duljina):
    # zvuk iz uzoraka, opet uz F_S: interpolacija niskopropusnim filtrom do f_s/2
    return signal.resample_poly(uzorci, korak(fs), 1)[:duljina]


def za_slusanje(x):
    # meki početak i kraj (bez klika), vrijednosti u rasponu −1 … 1
    x = np.array(x, dtype=float)
    rub = int(RUB_ZVUKA * FS)
    x[:rub] *= np.linspace(0, 1, rub)
    x[-rub:] *= np.linspace(1, 0, rub)
    return np.clip(x, -1, 1)


def tri_zvuka(x, fs):
    # izvorni zvuk i isti zvuk uzorkovan s f_s, bez filtra i nakon anti-aliasing filtra
    bez = rekonstruiraj(uzorkuj(x, fs), fs, len(x))
    s_filtrom = rekonstruiraj(uzorkuj(x, fs, filtar=True), fs, len(x))
    return za_slusanje(x), za_slusanje(bez), za_slusanje(s_filtrom)


def opis(f, fs):
    # retci ispisa za ton f uz f_s (postupak za prividnu frekvenciju iz poglavlja 12) i
    # pojačanje anti-aliasing filtra na f; svi su brojevi cijeli
    k, r, fa = prividna(f, fs)
    retci = [f"f = {broj(f)} Hz, f_s = {broj(fs)} Hz, Nyquistova frekvencija "
             f"f_s/2 = {broj(fs / 2)} Hz"]
    if f <= fs / 2:
        retci.append(f"f ≤ f_s/2: uzorci nose pravu frekvenciju, f_a = f = {broj(f)} Hz")
    else:
        retci.append(f"f > f_s/2: k = {k} (cijeli broj najbliži f/f_s), r = f − k·f_s = "
                     f"{broj(r)} Hz, prividna frekvencija f_a = |r| = {broj(fa)} Hz")
    g = pojacanje_db(fs, f)
    pojacanje = f"{broj(g)} dB" if g > -120 else "ispod −120 dB"
    retci.append(f"anti-aliasing filtar ({FILTAR_RED}. reda, lom {broj(lom_filtra(fs))} Hz): "
                 f"pojačanje na f je {pojacanje}")
    return retci


def tablica_savijanja():
    # primjer savijanja iz poglavlja 12 u kHz, uz provjeru da se uzorci kosinusa frekvencija
    # f i f_a poklapaju
    fs = SAVIJANJE_FS_KHZ
    n = np.arange(64)
    retci = [f"f_s = {broj(fs)} kHz, f_s/2 = {broj(fs / 2)} kHz",
             "f / kHz    k    r / kHz    f_a / kHz    knjiga: f_a / kHz    uzorci isti"]
    for f, knjiga in SAVIJANJE_FA_KNJIGA.items():
        k, r, fa = prividna(f, fs)
        isti = np.allclose(np.cos(2 * np.pi * f * n / fs), np.cos(2 * np.pi * fa * n / fs))
        retci.append(f"{f:>7}  {k:>3}  {broj(r):>9}  {broj(fa):>11}  {broj(knjiga):>19}"
                     f"    {'da' if isti else 'ne'}")
    return retci


def opis_kliznog():
    return [f"klizni ton od {broj(CHIRP_OD)} Hz do {broj(CHIRP_DO)} Hz u "
            f"{broj(CHIRP_TRAJANJE)} s; f_s = {broj(FS_CHIRP)} Hz, f_s/2 = {broj(FS_CHIRP / 2)} Hz"]


def opis_izbora():
    # primjer izbora f_s i anti-aliasing filtra iz poglavlja 12 i filtar ove bilježnice
    k, r, fa = prividna(IZBOR_FD, IZBOR_FS)
    return [f"f_s = {broj(IZBOR_FS)} Hz, smetnja {broj(IZBOR_FD)} Hz: k = {k}, prividna "
            f"frekvencija f_a = {broj(fa)} Hz (knjiga: {broj(IZBOR_FA_KNJIGA)} Hz)",
            f"kaskada dvaju RC krugova (lom {broj(IZBOR_LOM)} Hz) na smetnji: "
            f"{broj(rc_kaskada_db(IZBOR_FD, IZBOR_LOM), 1)} dB "
            f"(knjiga: {broj(IZBOR_PRIGUSENJE_KNJIGA, 1)} dB)",
            f"filtar ove bilježnice za istu f_s ({FILTAR_RED}. reda, lom "
            f"{broj(lom_filtra(IZBOR_FS))} Hz) na smetnji: "
            f"{broj(pojacanje_db(IZBOR_FS, IZBOR_FD))} dB"]


for redak in opis(F_POCETNA, FS_POCETNA):
    print(redak)

## 2. Savijanje frekvencije: primjer iz knjige u kilohercima

Frekvencija $\varOmega = 2\pi f/f_s$ ovisi samo o omjeru $f/f_s$. Primjer savijanja iz
poglavlja 12 ($f_s = 10$ Hz) zato vrijedi i u kilohercima: uz $f_s = 10$ kHz frekvencije 2, 6,
8, 12, 15, 18 i 23 kHz daju iste prividne frekvencije, samo u kHz. Tablica ih računa postupkom
iz poglavlja i uspoređuje s knjigom, a za svaku provjerava i da se uzorci tona frekvencije $f$ i
tona frekvencije $f_a$ poklapaju. Graf je trokutasta „pila” iz poglavlja: kružići su
frekvencije iz primjera, a uz svaki je frekvencija signala u kHz.

In [ ]:
for redak in tablica_savijanja():
    print(redak)
for f, knjiga in SAVIJANJE_FA_KNJIGA.items():
    assert prividna(f, SAVIJANJE_FS_KHZ)[2] == knjiga
print("Bilježnica i knjiga daju iste prividne frekvencije.")

fs = SAVIJANJE_FS_KHZ
f = np.linspace(0, 25, 2001)
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(f, prividna_niz(f, fs), color=TAMNA, lw=1.6, label="prividna frekvencija $f_a$")
for j in range(1, 6):
    ax.axvline(j * fs / 2, color=SIVA, ls=":", lw=0.9)
for v in SAVIJANJE_FA_KNJIGA:
    fa = prividna(v, fs)[2]
    ax.plot(v, fa, "o", ms=8, mfc="white", mec=BOJA, mew=1.8)
    ax.annotate(str(v), (v, fa), textcoords="offset points", xytext=(0, 11), ha="center",
                fontsize=9, bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none"))
ax.set_xlim(0, 25)
ax.set_ylim(-0.3, fs / 2 + 1.2)
ax.set_xlabel("frekvencija signala $f$ / kHz (točkasto: višekratnici $f_s/2$)")
ax.set_ylabel("$f_a$ / kHz")
ax.set_title(f"savijanje frekvencije uz $f_s$ = {fs} kHz", loc="left", fontsize=10)
ax.xaxis.set_major_formatter(zarez)
ax.yaxis.set_major_formatter(zarez)
fig.tight_layout()
plt.show()

## 3. Klizači: ton $f$ i frekvencija uzorkovanja $f_s$

Klizačem $f$ birate frekvenciju tona (od 100 Hz do 16 kHz), a klizačem $f_s$ frekvenciju
uzorkovanja. Lijevo je savijanje uz odabranu $f_s$: točka pokazuje kamo se ton preslika.
Desno je ton (svijetla crta) na prva 24 uzorka, uzorci (kružići) i ton prividne frekvencije
$f_a$ (crtkano) koji prolazi kroz iste uzorke. Ispod grafa ispis slijedi postupak za prividnu
frekvenciju i navodi pojačanje anti-aliasing filtra na frekvenciji tona.

Zatim slijede tri zvuka: izvorni ton, isti ton uzorkovan s $f_s$ bez filtra (čuje se $f_a$) i
uzorkovan nakon anti-aliasing filtra. Svi imaju isti faktor glasnoće (`GLASNOCA`), pa se čuje i
koliko filtar priguši ton.

Ako se klizači ne prikažu, pozovite funkciju izravno, npr. `pokus(5000, 6000)`.

In [ ]:
def pokus(f=F_POCETNA, fs=FS_POCETNA):
    k, r, fa = prividna(f, fs)
    fig, (ax_s, ax_t) = plt.subplots(1, 2, figsize=(13, 4.2),
                                     gridspec_kw={"width_ratios": [1, 1.35]})
    # savijanje uz odabranu f_s i točka za ton
    fv = np.linspace(0, F_MAX, 4001)
    ax_s.plot(fv / 1e3, prividna_niz(fv, fs) / 1e3, color=TAMNA, lw=1.4)
    for j in range(1, int(F_MAX // (fs / 2)) + 1):
        ax_s.axvline(j * fs / 2e3, color=SIVA, ls=":", lw=0.8)
    ax_s.plot(f / 1e3, fa / 1e3, "o", ms=9, mfc=BOJA, mec="white", mew=1.2)
    ax_s.set_xlim(0, F_MAX / 1e3 * 1.02)
    ax_s.set_ylim(0, fs / 2e3 * 1.15)
    ax_s.set_xlabel("frekvencija tona $f$ / kHz (točkasto: višekratnici $f_s/2$)")
    ax_s.set_ylabel("$f_a$ / kHz")
    ax_s.set_title(f"savijanje uz $f_s$ = {broj(fs)} Hz", loc="left", fontsize=10)

    # ton, uzorci i ton prividne frekvencije kroz iste uzorke
    t_n = np.arange(N_PRIKAZ) / fs
    kraj = t_n[-1]
    t = np.linspace(0, kraj, 4000 + int(20 * f * kraj))
    ax_t.plot(1e3 * t, np.cos(2 * np.pi * f * t), color=SIVA, lw=0.8, alpha=0.6,
              label=f"ton $f$ = {broj(f)} Hz")
    ax_t.plot(1e3 * t, np.cos(2 * np.pi * fa * t), color=BOJA, lw=2, ls="--",
              label=f"ton $f_a$ = {broj(fa)} Hz")
    ax_t.plot(1e3 * t_n, np.cos(2 * np.pi * f * t_n), "o", color=TAMNA, mfc="white", ms=6,
              mew=1.4, label="uzorci")
    ax_t.set_ylim(-1.25, 1.25)
    ax_t.set_xlabel("vrijeme $t$ / ms")
    ax_t.set_title(f"prvih {N_PRIKAZ} uzoraka", loc="left", fontsize=10)
    ax_t.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9)
    for ax in (ax_s, ax_t):
        ax.xaxis.set_major_formatter(zarez)
        ax.yaxis.set_major_formatter(zarez)
    fig.tight_layout()
    plt.show()
    for redak in opis(f, fs):
        print(redak)
    izvorni, bez, s_filtrom = tri_zvuka(ton(f), fs)
    print("Izvorni ton:")
    display(Audio(izvorni, rate=FS, normalize=False))
    print("Uzorkovan s f_s, bez filtra:")
    display(Audio(bez, rate=FS, normalize=False))
    print("Uzorkovan s f_s, nakon anti-aliasing filtra:")
    display(Audio(s_filtrom, rate=FS, normalize=False))


klizac = widgets.interactive(
    pokus,
    f=widgets.IntSlider(value=F_POCETNA, min=F_MIN, max=F_MAX, step=F_KORAK,
                        description="f / Hz", continuous_update=False,
                        layout=widgets.Layout(width="560px")),
    fs=widgets.SelectionSlider(options=FS_IZBOR, value=FS_POCETNA, description="f_s / Hz",
                               continuous_update=False, layout=widgets.Layout(width="560px")),
)
display(klizac)

## 4. Klizni ton koji pada natrag

Klizni ton jednoliko mijenja frekvenciju: ovdje raste od 100 Hz do 12 kHz u 6 s, a zatim ga
uzorkujemo s $f_s = 6000$ Hz (`FS_CHIRP`). Gore lijevo je frekvencija tona $f$ uz prividnu
frekvenciju $f_a$ prema formuli iz odjeljka 1. Ostale tri slike su *spektrogrami*: vodoravno je
vrijeme, okomito frekvencija, a tamnije je mjesto na kojem zvuk ima više snage (spektar
diskretnog signala obrađuje poglavlje 13). Izvorni ton je kosa crta. Uzorkovan bez filtra
prelama se na $f_s/2$ (točkasto), a uzorkovan nakon filtra slabi kad ton prijeđe lom filtra.
Slabi trag iznad $f_s/2$ na spektrogramima uzorkovanih tonova ne dolazi iz uzoraka: daju ga
interpolacijski filtar pri vraćanju zvuka na $F_S$ i prozor spektrograma, koji svaku frekvenciju
razmaže u uski pojas.

Poslušajte sva tri zvuka. Izvorni ton stalno raste; uzorkovani bez filtra raste samo dok ne
dosegne $f_s/2$, a onda „pada natrag”.

In [ ]:
for redak in opis_kliznog():
    print(redak)
izvorni, bez, s_filtrom = tri_zvuka(klizni_ton(), FS_CHIRP)

t = np.linspace(0, CHIRP_TRAJANJE, 1201)
fig, osi = plt.subplots(2, 2, figsize=(13, 7.2), sharex=True)
ax = osi[0, 0]
ax.plot(t, frekvencija_kliznog(t) / 1e3, color=SIVA, lw=3, alpha=0.7, label="frekvencija tona $f$")
ax.plot(t, prividna_niz(frekvencija_kliznog(t), FS_CHIRP) / 1e3, color=BOJA, lw=1.6, ls="--",
        label="prividna frekvencija $f_a$")
ax.axhline(FS_CHIRP / 2e3, color=TAMNA, ls=":", lw=1.2, label="$f_s/2$")
ax.set_ylim(0, CHIRP_DO / 1e3 * 1.05)
ax.set_title("frekvencija tona i prividna frekvencija", loc="left", fontsize=10)
ax.legend(loc="upper left", frameon=True, framealpha=0.9, fontsize=9)

_, _, S0 = signal.spectrogram(izvorni, fs=FS, nperseg=2048, noverlap=1024)
vrh_db = 10 * np.log10(S0.max())
for ax, zvuk, naslov, gornja in (
        (osi[0, 1], izvorni, "izvorni klizni ton", CHIRP_DO),
        (osi[1, 0], bez, f"uzorkovan s $f_s$ = {broj(FS_CHIRP)} Hz, bez filtra", FS_CHIRP),
        (osi[1, 1], s_filtrom, "uzorkovan nakon anti-aliasing filtra", FS_CHIRP)):
    fk, tk, S = signal.spectrogram(zvuk, fs=FS, nperseg=2048, noverlap=1024)
    ax.pcolormesh(tk, fk / 1e3, 10 * np.log10(S + 1e-20), cmap="gray_r", vmin=vrh_db - 60,
                  vmax=vrh_db, shading="auto", rasterized=True)
    if gornja == FS_CHIRP:
        ax.axhline(FS_CHIRP / 2e3, color=BOJA, ls=":", lw=1.4)
    ax.set_ylim(0, gornja / 1e3 * (1.05 if gornja == CHIRP_DO else 0.55))
    ax.set_title(naslov, loc="left", fontsize=10)
for ax in osi.flat:
    ax.set_ylabel("frekvencija / kHz")
    ax.xaxis.set_major_formatter(zarez)
    ax.yaxis.set_major_formatter(zarez)
for ax in osi[1]:
    ax.set_xlabel("vrijeme $t$ / s")
fig.tight_layout()
plt.show()

print("Izvorni klizni ton:")
display(Audio(izvorni, rate=FS, normalize=False))
print("Uzorkovan s f_s, bez filtra:")
display(Audio(bez, rate=FS, normalize=False))
print("Uzorkovan s f_s, nakon anti-aliasing filtra:")
display(Audio(s_filtrom, rate=FS, normalize=False))

## 5. Anti-aliasing filtar: koliko priguši

Koje frekvencije anti-aliasing filtar mora prigušiti i koliko, opisuju pravilo i postupak za
izbor frekvencije uzorkovanja u poglavlju 12. Ovdje samo usporedimo dva filtra na istoj smetnji.
U primjeru izbora $f_s$ i anti-aliasing filtra iz poglavlja 12 filtar je kaskada dvaju RC krugova
s lomom 250 Hz uz $f_s = 3$ kHz, a smetnja od 2950 Hz savija se u 50 Hz. Ćelija ispod računa prividnu frekvenciju
smetnje i pojačanje kaskade na njoj, uspoređuje ih s knjigom i dodaje pojačanje filtra koji
ova bilježnica koristi za istu $f_s$. Slika prikazuje pojačanje obaju filtara u dB.

In [ ]:
for redak in opis_izbora():
    print(redak)
assert prividna(IZBOR_FD, IZBOR_FS)[2] == IZBOR_FA_KNJIGA
assert round(float(rc_kaskada_db(IZBOR_FD, IZBOR_LOM)), 1) == IZBOR_PRIGUSENJE_KNJIGA
print("Bilježnica i knjiga daju iste brojke.")

f = np.logspace(1, np.log10(FS / 2), 800)
fig, ax = plt.subplots(figsize=(11, 4))
ax.semilogx(f, rc_kaskada_db(f, IZBOR_LOM), color=TAMNA, lw=1.8,
            label="kaskada dvaju RC krugova (primjer iz knjige)")
ax.semilogx(f, pojacanje_db(IZBOR_FS, f), color=BOJA, lw=1.8, ls="--",
            label=f"Butterworthov filtar {FILTAR_RED}. reda (ova bilježnica)")
ax.axvline(IZBOR_FS / 2, color=SIVA, ls=":", lw=1.4, label="$f_s/2$")
ax.axvline(IZBOR_FD, color=SIVA, ls="-.", lw=1.4, label="smetnja")
ax.set_ylim(-120, 6)
ax.set_xlabel("frekvencija $f$ / Hz")
ax.set_ylabel("pojačanje / dB")
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_title(f"anti-aliasing filtri uz $f_s$ = {broj(IZBOR_FS)} Hz", loc="left", fontsize=10)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9)
ax.yaxis.set_major_formatter(zarez)
fig.tight_layout()
plt.show()

## Pokušajte sami

1. Zadajte $f_s = 2400$ Hz i pomičite $f$ od 1000 do 1400 Hz u koracima od 50 Hz. Što čujete u
   inačici bez filtra? Ponovite uz $f_s = 24000$ Hz. Zašto se ista promjena $f$ čuje tako
   različito?
2. Nađite tri različite frekvencije $f$ koje uz $f_s = 6000$ Hz zvuče jednako. Provjerite ih
   postupkom za prividnu frekvenciju iz poglavlja 12 i sluhom.
3. Zadajte $f_s = 4800$ Hz i ton malo ispod $f_s/2$, npr. $f = 2300$ Hz, pa poslušajte sva tri
   zvuka. Koliko filtar priguši taj ton? Kako biste izabrali $f_s$ i lom filtra da ton te
   frekvencije prođe gotovo neprigušen, a alias tona iznad $f_s/2$ ne?
4. U postavkama zadajte `FS_CHIRP = 4800` i pokrenite sve ćelije. Prije slušanja predvidite
   koliko će puta uzorkovani klizni ton „pasti natrag” i na kojim frekvencijama tona.
5. U postavkama zadajte `FILTAR_RED = 2`, pa poslušajte zadani ton nakon filtra. Koliko se alias
   još čuje? Povežite to s pravilom o izboru frekvencije uzorkovanja u poglavlju 12.

---
Kod je pod licencom MIT (`LICENSE`), a tekst pod CC BY 4.0 (`LICENSE-sadrzaj.md`).